In [1]:
# %% Libraries 
import os
import sys
import pandas as pd
import plotnine as p9
from pathlib import Path
import pypalettes as pp
import matplotlib.colors as mcolors
import scanpy as sc
import numpy as np

# %% Set Main Directory
MAIN_DIR_NAME = "cosmx_cf_proseg"
MAIN_DIR = next(p for p in Path.cwd().parents if (p / MAIN_DIR_NAME).exists()) / MAIN_DIR_NAME

# add it to sys.path to set it as root directory
os.chdir(MAIN_DIR)

# %% Results Directories
PLOTS_DIR = Path("results/comb/spatial_polygons/FOXJ1")
print(PLOTS_DIR)

# %% CRATE code

# imports
import os
import sys
import pandas as pd
import plotnine as p9
from pathlib import Path
import pypalettes as pp
import matplotlib.colors as mcolors
import scanpy as sc
import geopandas as gpd

# create crate
def create_crate(adata_path, polygons_path, id_col):
    
    adata = sc.read_h5ad(adata_path)
    polygons = gpd.read_parquet(polygons_path)

    crate = {
        "adata": adata,
        "polygons": polygons,
        "id_col": id_col,
    }

    return crate

# subset crate
def subset_crate(crate, filter_mask):

    id_col = crate["id_col"]
    adata = crate["adata"]
    polygons = crate["polygons"]

    cells = adata.obs.index[filter_mask]
    sub_adata = adata[cells, :].copy()
    sub_polygons = polygons.loc[polygons[id_col].isin(cells)].copy()

    sub_crate = {
        "adata": sub_adata,
        "polygons": sub_polygons,
        "id_col": id_col
    }
    
    return sub_crate


results/comb/spatial_polygons/FOXJ1


In [2]:
def plot_polygons(
    crate,
    ann_var,
    fig_size=(20, 20),
    colors=None,
    cmap="viridis",
    background_color="black",
    use_raw=True,
    layer=None,
    log1p=False,
):
    import numpy as np
    import pandas as pd
    import plotnine as p9
    from scipy.sparse import issparse
    from pandas.api.types import is_numeric_dtype

    adata = crate["adata"]
    id_col = crate["id_col"]
    poly_df = crate["polygons"]

    # Read an observation annotation or gene expression
    if ann_var in adata.obs.columns:
        annotation = adata.obs[ann_var].copy()

    else:
        if use_raw:
            if adata.raw is None:
                raise ValueError("adata.raw is not available.")
            if layer is not None:
                raise ValueError(
                    "Use use_raw=False to read a layer. "
                    "adata.raw stores expression in .X."
                )
            source = adata.raw
        else:
            source = adata

        if ann_var not in source.var_names:
            location = "adata.raw.var_names" if use_raw else "adata.var_names"
            raise ValueError(f"Gene '{ann_var}' not found in {location}.")

        gene = source[:, [ann_var]]
        expression = gene.X if layer is None else gene.layers[layer]

        values = (
            expression.toarray() if issparse(expression)
            else np.asarray(expression)
        ).ravel()

        if log1p:
            values = np.log1p(values)

        annotation = pd.Series(
            values,
            index=adata.obs_names,
            name=ann_var,
        )

    # Use the ID column when available; otherwise use observation names
    cell_ids = (
        adata.obs[id_col].to_numpy()
        if id_col in adata.obs.columns
        else adata.obs_names.to_numpy()
    )

    meta_df = pd.DataFrame({id_col: cell_ids})
    meta_df["_plot_value"] = annotation.reset_index(drop=True)

    ann_poly = poly_df.merge(
        meta_df,
        on=id_col,
        how="left",
        validate="many_to_one",
    )

    # Fill scale
    if is_numeric_dtype(ann_poly["_plot_value"]):
        fill_scale = (
            p9.scale_fill_cmap(cmap_name=cmap, name=ann_var)
            if colors is None
            else p9.scale_fill_gradientn(colors=colors, name=ann_var)
        )
    else:
        fill_scale = (
            p9.scale_fill_hue(name=ann_var)
            if colors is None
            else p9.scale_fill_manual(values=colors, name=ann_var)
        )

    return (
        p9.ggplot(ann_poly)
        + p9.geom_map(
            p9.aes(geometry="geometry", fill="_plot_value"),
            color="white",
            size=0.1,
        )
        + p9.coord_fixed(1)
        + fill_scale
        + p9.theme(
            axis_line=p9.element_blank(),
            axis_text=p9.element_blank(),
            axis_ticks=p9.element_blank(),
            axis_title=p9.element_blank(),
            panel_background=p9.element_rect(fill=background_color),
            panel_grid_major=p9.element_blank(),
            panel_grid_minor=p9.element_blank(),
            figure_size=fig_size,
        )
    )

In [3]:
# %% Load adata and polygons

# Paths
OBJ_V = "cellcharter"
adata_path = Path(f"data/comb/h5ad/comb-{OBJ_V}.h5ad")
polygons_path = Path("data/comb/polygons/comb-polygons-light.parquet")
id_col = "cell_id"

# create crate
crate = create_crate(adata_path, polygons_path, id_col)

# %% get sample names
sample_names = crate["adata"].obs["sample_name"].unique()
# %% Set the annotation variable
VARS = ["FOXJ1"]



In [9]:

# %% Plot polygons

for sample in sample_names:

    print(sample)

    for ANN_VAR in VARS:

        print(ANN_VAR)

        # filter mask 
        filter_mask = crate["adata"].obs["sample_name"] == sample

        # subset crate
        sub_crate = subset_crate(crate, filter_mask)

        p = plot_polygons(sub_crate, ann_var=ANN_VAR, fig_size=(20,20))

        for ext in ['png', 'svg']:
            p.save(
                    filename= PLOTS_DIR / f'{sample}_{ANN_VAR}.{ext}',
                    dpi=900,
                    units='in'
                )

PWCF06
FOXJ1


/mnt/data/project0062/.conda/envs/rsc_25.12/lib/python3.13/site-packages/plotnine/ggplot.py:623: PlotnineWarning: Saving 20 x 20 in image.
/mnt/data/project0062/.conda/envs/rsc_25.12/lib/python3.13/site-packages/plotnine/ggplot.py:624: PlotnineWarning: Filename: results/comb/spatial_polygons/FOXJ1/PWCF06_FOXJ1.png
/mnt/data/project0062/.conda/envs/rsc_25.12/lib/python3.13/site-packages/plotnine/ggplot.py:623: PlotnineWarning: Saving 20 x 20 in image.
/mnt/data/project0062/.conda/envs/rsc_25.12/lib/python3.13/site-packages/plotnine/ggplot.py:624: PlotnineWarning: Filename: results/comb/spatial_polygons/FOXJ1/PWCF06_FOXJ1.svg


PWCF01_2
FOXJ1


/mnt/data/project0062/.conda/envs/rsc_25.12/lib/python3.13/site-packages/plotnine/ggplot.py:623: PlotnineWarning: Saving 20 x 20 in image.
/mnt/data/project0062/.conda/envs/rsc_25.12/lib/python3.13/site-packages/plotnine/ggplot.py:624: PlotnineWarning: Filename: results/comb/spatial_polygons/FOXJ1/PWCF01_2_FOXJ1.png
/mnt/data/project0062/.conda/envs/rsc_25.12/lib/python3.13/site-packages/plotnine/ggplot.py:623: PlotnineWarning: Saving 20 x 20 in image.
/mnt/data/project0062/.conda/envs/rsc_25.12/lib/python3.13/site-packages/plotnine/ggplot.py:624: PlotnineWarning: Filename: results/comb/spatial_polygons/FOXJ1/PWCF01_2_FOXJ1.svg


CTRL05
FOXJ1


...

/mnt/data/project0062/.conda/envs/rsc_25.12/lib/python3.13/site-packages/plotnine/ggplot.py:623: PlotnineWarning: Saving 20 x 20 in image.
/mnt/data/project0062/.conda/envs/rsc_25.12/lib/python3.13/site-packages/plotnine/ggplot.py:624: PlotnineWarning: Filename: results/comb/spatial_polygons/FOXJ1/PWCF05_FOXJ1.png
/mnt/data/project0062/.conda/envs/rsc_25.12/lib/python3.13/site-packages/plotnine/ggplot.py:623: PlotnineWarning: Saving 20 x 20 in image.
/mnt/data/project0062/.conda/envs/rsc_25.12/lib/python3.13/site-packages/plotnine/ggplot.py:624: PlotnineWarning: Filename: results/comb/spatial_polygons/FOXJ1/PWCF05_FOXJ1.svg


CTRL03
FOXJ1


/mnt/data/project0062/.conda/envs/rsc_25.12/lib/python3.13/site-packages/plotnine/ggplot.py:623: PlotnineWarning: Saving 20 x 20 in image.
/mnt/data/project0062/.conda/envs/rsc_25.12/lib/python3.13/site-packages/plotnine/ggplot.py:624: PlotnineWarning: Filename: results/comb/spatial_polygons/FOXJ1/CTRL03_FOXJ1.png
/mnt/data/project0062/.conda/envs/rsc_25.12/lib/python3.13/site-packages/plotnine/ggplot.py:623: PlotnineWarning: Saving 20 x 20 in image.
/mnt/data/project0062/.conda/envs/rsc_25.12/lib/python3.13/site-packages/plotnine/ggplot.py:624: PlotnineWarning: Filename: results/comb/spatial_polygons/FOXJ1/CTRL03_FOXJ1.svg


PWCF02
FOXJ1


/mnt/data/project0062/.conda/envs/rsc_25.12/lib/python3.13/site-packages/plotnine/ggplot.py:623: PlotnineWarning: Saving 20 x 20 in image.
/mnt/data/project0062/.conda/envs/rsc_25.12/lib/python3.13/site-packages/plotnine/ggplot.py:624: PlotnineWarning: Filename: results/comb/spatial_polygons/FOXJ1/PWCF02_FOXJ1.png
/mnt/data/project0062/.conda/envs/rsc_25.12/lib/python3.13/site-packages/plotnine/ggplot.py:623: PlotnineWarning: Saving 20 x 20 in image.
/mnt/data/project0062/.conda/envs/rsc_25.12/lib/python3.13/site-packages/plotnine/ggplot.py:624: PlotnineWarning: Filename: results/comb/spatial_polygons/FOXJ1/PWCF02_FOXJ1.svg
